# mmBVEX VDIF — decode, validate & DiFX-prep

Analysis notebook for the flat `.vdif` files written by `vdif_recorder` on the recording host.
Run this on the **workstation** (`your analysis workstation`), not on the recording host.

Stream: 8032-B VDIF frames (32-B header + 8000-B data), **2-bit complex** (offset binary),
single thread, 128,000 frames/s, 2048 Msps complex band.
Packer layout (per byte): `bits[1:0]=I, [3:2]=Q, [5:4]=I, [7:6]=Q`.

Install once: `pip install baseband astropy numpy matplotlib pandas`

In [ ]:
import os, glob, json, struct, collections, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- EDIT: point at one recorded file (and its directory for DiFX prep) ----
VDIF_PATH = 'bvex_20260626t200507Z_th0.vdif'   # the file the DiFX zero-baseline job used
if not os.path.exists(VDIF_PATH):
    _c = sorted(glob.glob('bvex_*.vdif')) or sorted(glob.glob('*.vdif'))
    if _c: VDIF_PATH = _c[0]                   # oldest = usually the validated capture
VDIF_DIR  = os.path.dirname(os.path.abspath(VDIF_PATH)) or '.'

FRAME, HDR, DATA = 8032, 32, 8000
SAMPLE_RATE_HZ   = 2048e6                # complex sample rate (Msps complex)
FRAMES_PER_SEC   = 128000
print('reading from', VDIF_PATH)

## 1. Self-contained reader (source of truth — matches the packer RTL exactly)
`baseband` is cross-checked later; this manual path is guaranteed correct for our format.

In [ ]:
def read_frames(path, max_frames=None):
    """Yield raw 8032-B VDIF frames from a flat .vdif file."""
    out = []
    with open(path, 'rb') as f:
        while True:
            fr = f.read(FRAME)
            if len(fr) < FRAME:
                break
            out.append(fr)
            if max_frames and len(out) >= max_frames:
                break
    return out

def parse_header(fr):
    w0, w1, w2, w3 = struct.unpack('<IIII', fr[:16])
    return dict(invalid=(w0>>31)&1, sec=w0&0x3FFFFFFF, frame=w1&0xFFFFFF,
                ref_epoch=(w1>>24)&0x3F, frame_len=(w2&0xFFFFFF)*8,
                log2nchan=(w2>>24)&0x1F, bps_m1=(w3>>26)&0x1F,
                data_type=(w3>>31)&1, thread=(w3>>16)&0x3FF, station=w3&0xFFFF)

# offset binary 00,01,10,11 -> signed 2-bit levels
LUT = np.array([-3.0, -1.0, 1.0, 3.0], dtype=np.float32)

def decode_complex(frame_bytes):
    """8032-B frame -> 16000 complex samples (I + jQ), per the packer RTL."""
    a = np.frombuffer(frame_bytes[HDR:HDR+DATA], dtype=np.uint8)
    f0, f1, f2, f3 = a & 3, (a>>2)&3, (a>>4)&3, (a>>6)&3
    Ic = np.empty(a.size*2, np.uint8); Ic[0::2]=f0; Ic[1::2]=f2   # I = sub-positions 0,2
    Qc = np.empty(a.size*2, np.uint8); Qc[0::2]=f1; Qc[1::2]=f3   # Q = sub-positions 1,3
    return LUT[Ic] + 1j*LUT[Qc]

def vdif_unix(ref_epoch, sec):
    y = 2000 + ref_epoch//2; mon = 7 if (ref_epoch % 2) else 1
    base = datetime.datetime(y, mon, 1, tzinfo=datetime.timezone.utc)
    return base.timestamp() + sec

frames = read_frames(VDIF_PATH, max_frames=400000)
print('read %d frames (%.1f MB)' % (len(frames), len(frames)*FRAME/1e6))

## 2. Header validation + frame# continuity

In [ ]:
hdrs = [parse_header(fr) for fr in frames]
assert hdrs, 'no frames read -- check VDIF_PATH'
h0 = hdrs[0]
print('frame_len=%d  bps-1=%d  data_type=%d  thread=%d  station=%d  ref_epoch=%d'
      % (h0['frame_len'], h0['bps_m1'], h0['data_type'], h0['thread'], h0['station'], h0['ref_epoch']))
ok = (h0['frame_len']==8032 and h0['bps_m1']==1 and h0['data_type']==1)
print('header', 'VALID (8032 B, 2-bit, complex)' if ok else 'UNEXPECTED -- check fields')
print('start UTC:', datetime.datetime.utcfromtimestamp(vdif_unix(h0['ref_epoch'], h0['sec'])).isoformat(), 'Z')

df = pd.DataFrame(hdrs)
gaps = dups = bad_reset = 0
for a, b in zip(hdrs, hdrs[1:]):
    if b['sec'] == a['sec']:
        if   b['frame'] == a['frame']+1: pass
        elif b['frame'] == a['frame']:   dups += 1
        elif b['frame'] >  a['frame']+1: gaps += b['frame'] - a['frame'] - 1
    elif b['sec'] == a['sec']+1:
        if b['frame'] != 0: bad_reset += 1
        gaps += (FRAMES_PER_SEC-1 - a['frame']) + b['frame']
maxf = df.groupby('sec')['frame'].agg(['min','max','count'])
print('\nseconds seen:', len(maxf))
print(maxf.head())
print('\ngaps=%d  dups=%d  bad_resets=%d  (want 0/0/0; max frame# per sec should be 127999)'
      % (gaps, dups, bad_reset))
# cross-check the recorder sidecar if present
sc = VDIF_PATH + '.json'
if os.path.exists(sc):
    print('\nsidecar:', json.load(open(sc)))

## 3. Decode to complex + 2-bit Van Vleck (I and Q separately)

In [ ]:
N = min(200, len(frames))                       # frames to histogram
z = np.concatenate([decode_complex(fr) for fr in frames[:N]])
I, Q = z.real, z.imag
print('decoded %d complex samples from %d frames' % (z.size, N))

def hist4(x):
    idx = ((x + 3) / 2).astype(int)             # -3,-1,1,3 -> 0,1,2,3
    c = np.bincount(idx, minlength=4).astype(float)
    return 100.0 * c / c.sum()

pI, pQ = hist4(I), hist4(Q)
print('level:        00     01     10     11   outer')
print('I  (%%):   %6.2f %6.2f %6.2f %6.2f   %.1f' % (*pI, pI[0]+pI[3]))
print('Q  (%%):   %6.2f %6.2f %6.2f %6.2f   %.1f' % (*pQ, pQ[0]+pQ[3]))
print('ideal VV: 16.40  33.60  33.60  16.40   32.7')

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for a, p, t in [(ax[0], pI, 'I'), (ax[1], pQ, 'Q')]:
    a.bar(range(4), p, color='steelblue')
    a.axhline(16.4, ls='--', c='r'); a.axhline(33.6, ls='--', c='r')
    a.set_xticks(range(4)); a.set_xticklabels(['00','01','10','11'])
    a.set_title('%s — 2-bit Van Vleck' % t); a.set_ylabel('%'); a.set_ylim(0, 45)
plt.tight_layout(); plt.show()

def is_vv(p):
    outer = p[0]+p[3]
    return (28 <= outer <= 40) and (p[1]+p[2] > outer) and not np.all(np.abs(p-25) < 4)
print('VERDICT:', 'genuine 2-bit complex (both I & Q Van Vleck)'
      if (is_vv(pI) and is_vv(pQ)) else 'check thresholds / tone / DC offset')

## 4. Power spectrum / bandpass (Welch-averaged)

In [ ]:
NFFT = 8192
nseg = z.size // NFFT
acc = np.zeros(NFFT)
win = np.hanning(NFFT)
for k in range(nseg):
    seg = z[k*NFFT:(k+1)*NFFT] * win
    acc += np.abs(np.fft.fftshift(np.fft.fft(seg)))**2
acc /= max(nseg, 1)
freq = np.fft.fftshift(np.fft.fftfreq(NFFT, 1.0/SAMPLE_RATE_HZ)) / 1e6   # MHz, baseband

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(freq, 10*np.log10(acc + 1e-9))
ax[0].set_xlabel('baseband frequency (MHz)'); ax[0].set_ylabel('power (dB)')
ax[0].set_title('Bandpass (%d x %d-pt segments)' % (nseg, NFFT)); ax[0].grid(True)
# time-domain total power
p_t = np.abs(z[:200000])**2
ax[1].plot(np.arange(p_t.size)/SAMPLE_RATE_HZ*1e6, p_t, lw=0.3)
ax[1].set_xlabel('time (us)'); ax[1].set_ylabel('|z|^2'); ax[1].set_title('Total power vs time')
plt.tight_layout(); plt.show()

## 5. Cross-check with `baseband` (the standard VLBI Python tool)
The manual decoder above is the reference; this confirms `baseband` agrees.

In [ ]:
try:
    import baseband.vdif as bvdif
    import astropy.units as u
    fh = bvdif.open(VDIF_PATH, 'rs', sample_rate=2048*u.MHz)
    print('baseband: start =', fh.start_time.isot, ' rate =', fh.sample_rate,
          ' complex =', fh.complex_data)
    d = fh.read(16000)
    print('baseband decoded', d.shape, d.dtype, '| first sample', d.flat[0])
    print('manual   first sample', decode_complex(frames[0])[0],
          '(scale/convention may differ; shape & stats are what matter)')
except Exception as e:
    print('baseband path note:', repr(e))
    print('-> use the manual decoder (cells above); it matches the packer RTL exactly.')

## 6. DiFX prep — emit a `.v2d` + `filelist` skeleton
Scans `VDIF_DIR` for `.vdif` files (+ sidecars) and writes a DiFX `filelist`
(`path startMJD stopMJD`) plus a `.v2d` template. **You still supply a `.vex`** with
station position, sky frequency/sideband, polarisation and clock model.

**DiFX-verified conventions (the workstation, DiFX 2.8.1, 2026-07-01):**
- mark5access format string = `VDIFC_8000-8192-1-2` (m5d decode matches the packer RTL
  sample-for-sample); v2d/vex format = **`VDIFD/8032/2`** — complex **double-sideband**
  (RFDC fine mixer puts band center at DC). `VDIFC` = SSB complex, NOT ours.
- vex `$FREQ` must follow the real-Nyquist convention: `sample_rate = 2048.0 Ms/sec`,
  `chan_def` bandwidth `1024.00 MHz` (vex2difx enforces BW ≤ rate/2); complex-ness comes
  from the format string only.
- Single-station autocorr jobs need `minSubarray = 1` in the v2d (default 2 drops the scan).

In [ ]:
def mjd(unix_ts): return unix_ts/86400.0 + 40587.0

files = sorted(glob.glob(os.path.join(VDIF_DIR, '*.vdif')))
lines = []
for vf in files:
    scf = vf + '.json'
    if os.path.exists(scf):
        m = json.load(open(scf))
        start = vdif_unix(m['ref_epoch'], m['first_sec'])
        stop  = vdif_unix(m['ref_epoch'], m['last_sec'] + 1)
    else:
        fr = read_frames(vf, 1); h = parse_header(fr[0])
        start = vdif_unix(h['ref_epoch'], h['sec']); stop = start + 10
    lines.append('%s %.9f %.9f' % (os.path.abspath(vf), mjd(start), mjd(stop)))

if not os.path.exists('bvex.filelist'):          # gen_vex.py's version is authoritative
    with open('bvex.filelist', 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print('wrote bvex.filelist (%d files)' % len(files))
else:
    print('bvex.filelist exists (gen_vex.py) -- not overwriting')

v2d = '''# bvex.v2d SKELETON -- DiFX config for the RFSoC 2-bit complex VDIF.
# Conventions VERIFIED against DiFX 2.8.1 (the workstation, 2026-07-01). The WORKING config is written
# by gen_vex.py (bvex.vex + bvex.v2d, incl. the 2-station zero-baseline test) -- use that.
vex = bvex.vex
minLength   = 1
minSubarray = 1                    # allow single-station (autocorrelation) jobs
singleScan  = True

ANTENNA Bv
{
    filelist      = bvex.filelist
    format        = VDIFD/8032/2   # complex DOUBLE-sideband VDIF (VDIFC would be SSB)
    phaseCalInt   = 0
    toneSelection = none
}

SETUP default
{
    tInt       = 1.0               # integration time (s)
    FFTSpecRes = 4.0
    specRes    = 4.0               # MHz per channel
}
'''
with open('bvex.v2d.skeleton', 'w') as f:   # NOT bvex.v2d -- don't clobber the working config
    f.write(v2d)
print('wrote bvex.v2d.skeleton (reference). Working flow: python3 gen_vex.py && vex2difx bvex.v2d')

# 7. DiFX ZERO-BASELINE VALIDATION — fringes, delay model & timing (run 2026-07-01)

**The test that proves the data is correlator-ready.** The recording
`bvex_20260626t200507Z_th0.vdif` (real hardware data: RFSoC → 100GbE → AF_XDP recorder) was
registered in DiFX **twice**, as two fake stations **BV** and **BW** placed **1 m apart** (ECEF X
offset) near Kingston. DiFX then treats them as a real interferometer baseline: it applies each
station's own geometric delay/fringe-rotation model from CALC 11 and cross-correlates.

Because the two datastreams are *byte-identical*, every observable has an exact prediction:

| Observable | Prediction | Why |
|---|---|---|
| zero-lag correlation coefficient | ≈ 1 (minus ~1–2 % processing loss) | same bits in both streams |
| cross-power phase slope across the band | exactly the **1-m model delay** (≈ 2.93 ns) | DiFX shifts BW by *its* model, BV by *its* model; the difference survives |
| lag-space fringe | single sharp peak at that delay | Fourier pair of the phase slope |
| phase residual after removing the fitted delay | ≈ 0° | no other physics in the loop |
| autocorrs BV vs BW | identical, real-valued | same data |

So a "boring" match here simultaneously exercises: VDIF decode inside mpifxcorr, the complex-DSB
sample handling, timestamp → correlator-time mapping, the CALC delay model, fringe rotation, and
the FX correlation itself. A failure anywhere shows up as decorrelation or wrong delay.

**Toolchain**: DiFX 2.8.1 (userspace build on the workstation, `~/difx`) — `gen_vex.py` → `vex2difx` →
`difxcalc` → `mpifxcorr` (`run_mpifx.sh`) → SWIN output `bvex_1.difx/`.
**Config**: format `VDIFD/8032/2` (2-bit complex **double-sideband**), `DATA SAMPLING: COMPLEX_DSB`,
band 3072 MHz USB × 1024 MHz, 256 × 4 MHz channels, tInt = 1 s, scan `2026y177d20h05m07s` + 3 s.

Prereqs in this directory: `bvex_1.difx/DIFX_*` (visibilities), `bvex_1.im` (delay polynomials),
`bvex_1.input`. Reproduce with: `python3 gen_vex.py && vex2difx bvex.v2d && difxcalc bvex_1.calc
&& ./run_mpifx.sh`.

In [ ]:
# --- 7.1 Load the DiFX (SWIN) visibilities + record inventory / timing check -------------
# SWIN binary record: sync 0xFF00FF00, version, baseline#, mjd, sec(double), config, source,
# freq, polpair[2], pulsarbin, weight(double), uvw[3 doubles], then nchan complex64 vis.
# Baseline number = 256*(ant1+1) + (ant2+1):  257=BVxBV auto, 514=BWxBW auto, 258=BVxBW cross.
DIFX_GLOB = 'bvex_1.difx/DIFX_*'

def load_swin(pattern):
    SYNC = 0xFF00FF00
    recs = {}
    for fn in sorted(glob.glob(pattern)):
        buf = open(fn, 'rb').read()
        off = 0
        while off + 74 <= len(buf):
            sync, = struct.unpack_from('<I', buf, off)
            if sync != SYNC:
                off += 1; continue
            bl, rmjd = struct.unpack_from('<ii', buf, off + 8)
            rsec, = struct.unpack_from('<d', buf, off + 16)
            pol = buf[off + 36:off + 38].decode()
            wgt, = struct.unpack_from('<d', buf, off + 42)
            uvw = struct.unpack_from('<3d', buf, off + 50)
            vo = off + 74
            nxt = buf.find(struct.pack('<I', SYNC), vo)
            end = nxt if nxt != -1 else len(buf)
            v = np.frombuffer(buf, dtype=np.complex64, count=(end - vo) // 8, offset=vo)
            recs.setdefault((bl, pol), []).append(dict(mjd=rmjd, sec=rsec, wgt=wgt, uvw=uvw, vis=v.copy()))
            off = end
    return recs

recs = load_swin(DIFX_GLOB)
assert recs, 'no SWIN records -- run the DiFX job first (see section header)'
NCH = next(iter(recs.values()))[0]['vis'].size
BW_MHZ = 1024.0; F0_MHZ = 3072.0
chan_mhz = F0_MHZ + (np.arange(NCH) + 0.5) * BW_MHZ / NCH   # channel centres, sky freq

print('record inventory:')
for k in sorted(recs):
    r = recs[k]
    name = {257: 'BV auto', 514: 'BW auto', 258: 'BV x BW cross'}.get(k[0], '?')
    print('  bl %3d (%s) pol %s : %d integrations x %d ch, weights %s'
          % (k[0], name, k[1], len(r), r[0]['vis'].size, [round(x['wgt'], 4) for x in r]))

print('\nintegration timestamps (correlator time) vs the VDIF data time:')
for r in recs[(258, 'RR')]:
    ut = (r['sec']) / 3600.0
    print('  MJD %d sec %8.2f  = %02d:%02d:%05.2f UT  (VDIF data seconds are 72307..72309 = 20:05:07..09)'
          % (r['mjd'], r['sec'], int(ut), int(ut % 1 * 60), (ut * 60) % 1 * 60))
print('\n-> DiFX placed each 1-s integration exactly on the VDIF/PPS second boundaries: the')
print('   ref_epoch/seconds/frame# written by the FPGA map to correlator time with no offset.')

In [ ]:
# --- 7.2 The CALC-11 delay model (bvex_1.im): what DiFX applied to each station -----------
# difxcalc writes, per scan/poly, per source, per antenna, 5th-order delay polynomials in us.
# The BV-BW model delay difference is the ground truth the measured fringe must reproduce.
import re as _re

def read_im_polys(path='bvex_1.im'):
    polys = {}   # (src, ant) -> {DELAY: coeffs, DRY:..., WET:...}; first poly interval only
    meta = {}
    seen_poly = 0
    for line in open(path):
        key, _, val = line.partition(':')
        key = key.strip()
        if key in ('SCAN 0 POLY 0 MJD', 'SCAN 0 POLY 0 SEC', 'INTERVAL (SECS)', 'POLYNOMIAL ORDER'):
            meta[key] = float(val)
        if key.startswith('SCAN 0 POLY 1'):
            seen_poly = 1
        m = _re.match(r'SRC (\d+) ANT (\d+) (DELAY \(us\)|DRY \(us\)|WET \(us\)|AZ|EL GEOM|U \(m\)|V \(m\)|W \(m\))', key)
        if m and not seen_poly:
            polys[(int(m.group(1)), int(m.group(2)), m.group(3))] = np.array([float(x) for x in val.split()])
    return polys, meta

polys, meta = read_im_polys()
print('.im: poly start MJD %d sec %d, interval %d s, order %d'
      % (meta['SCAN 0 POLY 0 MJD'], meta['SCAN 0 POLY 0 SEC'], meta['INTERVAL (SECS)'], meta['POLYNOMIAL ORDER']))

d0 = polys[(0, 0, 'DELAY (us)')]; d1 = polys[(0, 1, 'DELAY (us)')]
tpoly0 = meta['SCAN 0 POLY 0 SEC']
print('\ntotal geometric delay at poly start (BOTH ~10.67 ms: light-time to the geocenter-ish ref):')
print('  BV : %.9f us    BW : %.9f us' % (d0[0], d1[0]))
ddiff = d1 - d0                        # BW - BV, us
print('\nBW-BV model DIFFERENCE (the 1-m baseline seen toward 3C273):')
print('  delay  = %.6f ns' % (ddiff[0] * 1e3))
print('  rate   = %.6f ps/s' % (ddiff[1] * 1e6))
print('  (1 m / c = 3.336 ns is the maximum; the projection toward the source gives the value above)')

def model_delay_diff_ns(t_sec):
    """BW-BV delay difference (ns) at t_sec seconds after poly start."""
    dt = t_sec - tpoly0
    return np.polyval(ddiff[::-1], dt) * 1e3

for r in recs[(258, 'RR')]:
    print('  at integration midpoint sec %.2f: model delay diff = %.6f ns' % (r['sec'], model_delay_diff_ns(r['sec'])))
MODEL_DELAY_NS = model_delay_diff_ns(recs[(258, 'RR')][1]['sec'])

In [ ]:
# --- 7.3 Autocorrelation bandpasses (BV, BW) ----------------------------------------------
ac1 = np.mean([r['vis'] for r in recs[(257, 'RR')]], axis=0)
ac2 = np.mean([r['vis'] for r in recs[(514, 'RR')]], axis=0)
xc  = np.mean([r['vis'] for r in recs[(258, 'RR')]], axis=0)

print('autocorrs are real-valued: |imag|/|real| = %.2e (BV), %.2e (BW)'
      % (np.abs(ac1.imag).mean() / np.abs(ac1.real).mean() + 1e-30,
         np.abs(ac2.imag).mean() / np.abs(ac2.real).mean() + 1e-30))
print('BV vs BW autocorr identical (same data): max fractional diff = %.2e'
      % np.max(np.abs(ac1.real - ac2.real) / np.abs(ac1.real)))

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.6))
ax[0].plot(chan_mhz, ac1.real, lw=1, label='BV')
ax[0].plot(chan_mhz, ac2.real, lw=1, ls='--', label='BW')
ax[0].set_xlabel('sky frequency (MHz)'); ax[0].set_ylabel('autocorr power')
ax[0].set_title('Station autocorrelation bandpass (1024 MHz USB half of the DSB band)')
ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].semilogy(chan_mhz, ac1.real, lw=1)
ax[1].annotate('DC spike (ch 0 = complex-baseband DC)\n= the known Q-channel DC offset',
               xy=(chan_mhz[0], ac1.real[0]), xytext=(chan_mhz[40], ac1.real[0] * .6),
               arrowprops=dict(arrowstyle='->'), fontsize=8)
ax[1].set_xlabel('sky frequency (MHz)'); ax[1].set_title('BV autocorr (log) — DC-offset spike visible')
ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print('The flat noise bandpass + isolated DC spike is exactly what the noise-only input +')
print('measured Q DC offset (outer levels 20.9%%/13.6%%) predicts. Band median power = %.2f, DC = %.1f'
      % (np.median(ac1.real), ac1.real[0]))

In [ ]:
# --- 7.4 THE FRINGES: cross-power phase, lag-space fringe peak, delay fit vs model --------
f_off = (np.arange(NCH) + 0.5) * BW_MHZ / NCH * 1e6      # Hz offset within the band

# (a) raw cross phase across the band -- the fringe "winding" from the 1-m model delay
ph_raw = np.angle(xc)

# (b) fringe fit, frequency domain: unwrapped-phase slope -> residual delay
ph_un = np.unwrap(ph_raw)
A = np.vstack([f_off, np.ones(NCH)]).T
slope, icpt = np.linalg.lstsq(A, ph_un, rcond=None)[0]
tau_fit_ns = slope / (2 * np.pi) * 1e9

# (c) fringe fit, lag domain: zero-padded IFFT of the cross spectrum -> delay peak
PAD = 32
lagspec = np.fft.ifft(xc, NCH * PAD)
lags_ns = np.fft.fftfreq(NCH * PAD, d=(BW_MHZ * 1e6 / NCH)) * 1e9
order = np.argsort(lags_ns)
lag_amp = np.abs(lagspec)[order]; lags_ns = lags_ns[order]
tau_peak_ns = lags_ns[np.argmax(lag_amp)]

# (d) remove the fitted delay -> residual phase + coherence
xc_corr = xc * np.exp(-1j * (slope * f_off + icpt))
resid_deg = np.degrees(np.angle(xc_corr))
coh = np.abs(xc) / np.sqrt(np.abs(ac1.real) * np.abs(ac2.real))

# sign of the delay depends on the antenna-reference / FFT convention; the physics test is the
# MAGNITUDE (and that phase-slope fit and lag peak agree with each other and with CALC 11)
agree_ps = abs(abs(tau_fit_ns) - abs(MODEL_DELAY_NS)) * 1e3
print('FRINGE FIT (BV x BW, 3 s coherent average):')
print('  |delay|, phase-slope fit : %.4f ns' % abs(tau_fit_ns))
print('  |delay|, lag-space peak  : %.4f ns' % abs(tau_peak_ns))
print('  |delay|, CALC-11 model   : %.4f ns   -> agreement %.1f ps (%.4f%% of the 1-m delay)'
      % (abs(MODEL_DELAY_NS), agree_ps, 100 * agree_ps / 1e3 / abs(MODEL_DELAY_NS)))
print('  residual after delay removal: %.3f deg rms' % resid_deg.std())
print('  correlation coefficient  : mean %.4f  min %.4f  (~1-2%% below 1 is normal FX processing loss)'
      % (coh.mean(), coh.min()))
print('  phase winding across the band: %.2f turns (= %.3f ns x %.0f MHz)'
      % (abs(tau_fit_ns) * 1e-9 * BW_MHZ * 1e6, abs(tau_fit_ns), BW_MHZ))

fig, ax = plt.subplots(2, 2, figsize=(12.5, 7))
ax[0, 0].plot(chan_mhz, np.degrees(ph_raw), '.', ms=3)
ax[0, 0].set_ylim(-180, 180); ax[0, 0].set_xlabel('sky frequency (MHz)'); ax[0, 0].set_ylabel('phase (deg)')
ax[0, 0].set_title('Raw cross-power phase — %.1f fringe turns across the band' % (abs(tau_fit_ns) * BW_MHZ / 1e3))
ax[0, 0].grid(alpha=.3)
w = np.abs(lags_ns) < 25
ax[0, 1].plot(lags_ns[w], lag_amp[w] / lag_amp.max(), lw=1)
ax[0, 1].axvline(np.sign(tau_peak_ns) * abs(MODEL_DELAY_NS), color='r', ls='--', lw=1,
                 label='CALC-11 model |delay| = %.3f ns' % abs(MODEL_DELAY_NS))
ax[0, 1].set_xlabel('lag (ns)'); ax[0, 1].set_ylabel('normalised fringe amplitude')
ax[0, 1].set_title('Lag-space fringe: single peak at the model delay'); ax[0, 1].legend(); ax[0, 1].grid(alpha=.3)
ax[1, 0].plot(chan_mhz, resid_deg, '.', ms=3)
ax[1, 0].set_xlabel('sky frequency (MHz)'); ax[1, 0].set_ylabel('phase (deg)')
ax[1, 0].set_title('Residual phase after removing the fitted delay (rms %.3f deg)' % resid_deg.std())
ax[1, 0].grid(alpha=.3)
ax[1, 1].plot(chan_mhz, coh, lw=1)
ax[1, 1].axhline(1.0, color='k', ls=':', lw=1); ax[1, 1].set_ylim(0.9, 1.02)
ax[1, 1].set_xlabel('sky frequency (MHz)'); ax[1, 1].set_ylabel('|xc| / sqrt(ac1*ac2)')
ax[1, 1].set_title('Zero-baseline correlation coefficient (mean %.4f)' % coh.mean())
ax[1, 1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('difx_fringe_suite.png', dpi=110); plt.show()

In [ ]:
# --- 7.5 Per-integration fringe solutions: timing / coherence stability over the scan -----
# Each 1-s integration is fitted independently. For byte-identical streams the delay must be
# rock-stable at the model value and the integrations mutually coherent.
rows = []
for r in recs[(258, 'RR')]:
    v = r['vis']
    pu = np.unwrap(np.angle(v))
    sl, ic = np.linalg.lstsq(A, pu, rcond=None)[0]
    vr = v * np.exp(-1j * (sl * f_off + ic))
    rows.append(dict(sec=r['sec'], wgt=r['wgt'],
                     tau_ns=abs(sl / (2 * np.pi) * 1e9),
                     model_ns=abs(model_delay_diff_ns(r['sec'])),
                     amp=np.abs(v).mean(),
                     ph0_deg=np.degrees(ic) % 360,
                     resid_deg=np.degrees(np.angle(vr)).std()))

print('per-integration fringe fits:')
print('  t_mid (UT sec)   weight   |tau_fit| ns   |tau_model| ns   diff ps   mean amp   resid deg rms')
for w in rows:
    print('  %12.2f   %6.3f   %12.4f   %13.4f   %7.1f   %8.3f   %10.3f'
          % (w['sec'], w['wgt'], w['tau_ns'], w['model_ns'],
             (w['tau_ns'] - w['model_ns']) * 1e3, w['amp'], w['resid_deg']))
taus = np.array([w['tau_ns'] for w in rows])
print('\n  delay stability over the scan: %.2f ps rms (model drift over 3 s: %.3f ps)'
      % (taus.std() * 1e3, abs(model_delay_diff_ns(rows[-1]['sec']) - model_delay_diff_ns(rows[0]['sec'])) * 1e3))

# integration-to-integration coherence: identical data + same model => aligned phases
ph_t = [np.angle(np.vdot(recs[(258, 'RR')][0]['vis'], r['vis'])) for r in recs[(258, 'RR')]]
print('  inter-integration phase (vs first integration): %s deg'
      % ', '.join('%.3f' % np.degrees(p) for p in ph_t))

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.4))
t = [w['sec'] - rows[0]['sec'] for w in rows]
ax[0].plot(t, taus * 1e3, 'o-', label='fitted |delay|')
ax[0].plot(t, [w['model_ns'] * 1e3 for w in rows], 'r--', label='CALC-11 model')
ax[0].set_xlabel('time into scan (s)'); ax[0].set_ylabel('delay (ps)')
ax[0].set_title('Delay vs time: fringe tracks the model'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(t, [w['amp'] for w in rows], 'o-')
ax[1].set_xlabel('time into scan (s)'); ax[1].set_ylabel('mean fringe amplitude')
ax[1].set_ylim(0, max(w['amp'] for w in rows) * 1.2)
ax[1].set_title('Fringe amplitude per integration (no coherence loss over the scan)')
ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('difx_fringe_timing.png', dpi=110); plt.show()

In [ ]:
# --- 7.6 VERDICT: DiFX-readiness scorecard -------------------------------------------------
wgts = [r['wgt'] for r in recs[(258, 'RR')]]
checks = [
    ('mpifxcorr ran to completion on the raw recording', min(wgts) > 0.98,
     '3 x 1-s integrations, exit 0, weights %s (edge subints slightly clipped -- normal)'
     % ', '.join('%.3f' % w for w in wgts)),
    ('integration timestamps land on VDIF/PPS second boundaries',
     all(abs((r['sec'] - 0.5) % 1.0) < 1e-6 for r in recs[(258, 'RR')]),
     'FPGA ref_epoch/sec/frame# -> correlator time, no offset'),
    ('autocorr bandpass sane on both stations',
     bool(np.all(ac1.real > 0)) and np.abs(ac1.imag).max() < 1e-3 * np.abs(ac1.real).max(),
     'real, positive, flat noise band + known DC-offset spike'),
    ('zero-baseline correlation coefficient ~1', coh.mean() > 0.95,
     'measured %.4f (2-bit FX processing loss only)' % coh.mean()),
    ('fringe delay matches the CALC-11 model', agree_ps < 50,
     '|fit - model| = %.1f ps on a %.3f ns delay' % (agree_ps, abs(MODEL_DELAY_NS))),
    ('phase residual after delay removal ~0', resid_deg.std() < 1.0,
     '%.3f deg rms across 256 channels' % resid_deg.std()),
    ('delay stable across integrations', taus.std() * 1e3 < 50,
     '%.2f ps rms over the scan' % (taus.std() * 1e3)),
]
print('DiFX ZERO-BASELINE VALIDATION — SCORECARD')
print('=' * 78)
ok_all = True
for name, ok, detail in checks:
    ok_all &= bool(ok)
    print('  [%s]  %s' % ('PASS' if ok else 'FAIL', name))
    print('          %s' % detail)
print('=' * 78)
print('OVERALL: %s' % ('PASS — the 2-bit complex VDIF stream is DiFX-ready / correlator-ready.'
                       if ok_all else 'ATTENTION — see failed rows above'))
print()
print('Still pending (hardware, not format): tone-injection recording for sideband sense;')
print('Q-channel DC offset tuning; real station code in inp2_st_id; real coords + GPS clock')
print('in the .vex for a genuine two-station fringe test.')